# RAG Pipeline: AI Research Papers
## 1. Load & Inspect

In [1]:
import os, logging
from pypdf import PdfReader

logging.getLogger("pypdf").setLevel(logging.ERROR)
RAW_DIR = "../data/raw"
docs = []
failed = []

for name in sorted(os.listdir(RAW_DIR)):
    try:
        reader = PdfReader(os.path.join(RAW_DIR, name))
        for i, page in enumerate(reader.pages):
            text = page.extract_text() or ""
            if text.strip():
                docs.append({"source": name, "page": i + 1, "text": text})
    except Exception as e:
        failed.append((name, str(e)))

print("Files:", len(os.listdir(RAW_DIR)))
print("Pages with text:", len(docs))
print("Failed:", failed)            

Files: 32
Pages with text: 1012
Failed: []


### Data Summary
- **Documents:** 32 PDF files (arXiv research papers on AI topics)
- **Pages:** 1012 pages, all with extractable text
- **Failed to parse:** none
- **Need OCR:** none
- **Messy content handled:** ligatures (ﬁ ﬂ ﬃ) normalized with NFKC, hyphenated line-break words rejoined, broken line breaks removed
- **Not handled:** page headers (e.g. "6 Moerland et al.") and math symbols remain in the text

In [2]:
print(docs[5]["source"], "page", docs[5]["page"])
print(docs[5]["text"][:1500])

1705.05172v1.pdf page 6
6 Moerland et al.
formally introduce in the next section. There are however other machine learning
implementations that incorporate emotions. Some examples include agents based
on evolutionary neural networks (Parisi and Petrosino, 2010), the free-energy prin-
ciple (Joﬃly and Coricelli, 2013), Bayesian models (Antos and Pfeﬀer, 2011) or
entropy (Belavkin, 2004).
Finally, we want to stress that the focus of this review is on agent emotion,
i.e. how it is elicited and may inﬂuence the agent’s learning loop. A related but
clearly distinct topic is how human emotion may act as a teaching signal for this
loop. Broekens (2007) showed human emotional feedback speeds up agent learning
in a grid-world task compared to a baseline agent. There are a few other examples
in this direction (Hasson et al, 2011; Moussa and Magnenat-Thalmann, 2013),
but in general the literature of emotion as a teaching signal is limited. Although
the way in which humans actually tend to provide

In [3]:
import re
from collections import Counter

by_src = {}
for d in docs:
    by_src.setdefault(d["source"], []).append(d)

def norm(l):
    return re.sub(r"\d+", "#", l.strip().lower())

for src, pages in by_src.items():
    cnt = Counter()
    for p in pages:
        lines = [l for l in p["text"].split("\n") if l.strip()]
        for l in {norm(x) for x in lines[:3] + lines[-3:]}:
            cnt[l] += 1
    repeated = {l for l, c in cnt.items() if c >= max(3, 0.3 * len(pages)) and len(l) < 80}
    for p in pages:
        p["text"] = "\n".join(l for l in p["text"].split("\n") if norm(l) not in repeated)

In [4]:
import re, unicodedata

def clean_text(text):
    text = unicodedata.normalize("NFKC", text)   
    text = re.sub(r"-\n(?=[a-z])", "", text)   
    text = re.sub(r"\n(?!\n)", " ", text)       
    text = re.sub(r"\s+", " ", text)       
    return text.strip()

for d in docs:
    d["text"] = clean_text(d["text"])

print(docs[5]["text"][:700])

formally introduce in the next section. There are however other machine learning implementations that incorporate emotions. Some examples include agents based on evolutionary neural networks (Parisi and Petrosino, 2010), the free-energy principle (Joffily and Coricelli, 2013), Bayesian models (Antos and Pfeffer, 2011) or entropy (Belavkin, 2004). Finally, we want to stress that the focus of this review is on agent emotion, i.e. how it is elicited and may influence the agent’s learning loop. A related but clearly distinct topic is how human emotion may act as a teaching signal for this loop. Broekens (2007) showed human emotional feedback speeds up agent learning in a grid-world task compared


## 2. Chunking Strategy
RecursiveCharacterTextSplitter (LangChain): 1000 characters per chunk, 200 characters overlap (20%).
- It tries to split at sentence boundaries first, then words, so chunks are less likely to cut mid-sentence.
- 1000 chars (~250 tokens) is about the input limit of the embedding model (all-MiniLM-L6-v2, 256 tokens).
- The overlap keeps context for sentences that fall on a chunk boundary.
- Splitting per page keeps `source` and `page` metadata on every chunk, so answers can cite document + page.

In [5]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

documents = [
    Document(page_content=d["text"], metadata={"source": d["source"], "page": d["page"]})
    for d in docs
]

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ". ", " ", ""],
    keep_separator="end",
)

chunks = splitter.split_documents(documents)
chunks = [c for c in chunks if len(c.page_content) >= 100]   # ignore the small pieces

print("Total chunks:", len(chunks))
print(chunks[10].metadata)
print(chunks[10].page_content)

Total chunks: 4017
{'source': '1705.05172v1.pdf', 'page': 3}
Additionally, the comparison of evaluation criteria is presented in (Section 7). The survey ends with a general discussion of our findings, highlights some important problems and indicates future directions in this field (Section 8). 2 Background As many papers included in this survey build upon psychological (2.1) and neuroscientific (2.2) theories of emotion, this section provides a high-level overview of these fields. Subsequently, we position our work in the computer science and machine learning community (2.3). We conclude these preliminaries by formally introducing computational reinforcement learning (2.4). 2.1 Psychology We discuss three dominant psychological emotion theories: categorical, dimensional, and componential theories (see also Lisetti and Hudlicka (2014)). Categorical emotion theory assumes there is a set of discrete emotions forming the ‘basic’ emotions.


### Removing reference sections
Reference lists and author lists match many question words but contain no answers, so they crowded out real content in retrieval. Chunks that look like references (many years, "arXiv", or long author lists) are removed before embedding.

In [6]:
def author_list_score(text):
    return len(re.findall(r"[A-Z][a-zé'’-]+, (?:[A-Z] ?\. ?)+", text))

def looks_like_references(text):
    years = len(re.findall(r"\b(?:19|20)\d{2}\b", text))
    ref_markers = len(re.findall(r"\[\d{1,3}\]|\[[A-Z]{2,}\d{2}\]|pp\.\s*\d", text))
    return (
        (years >= 5 and ref_markers >= 2)
        or text.lower().count("arxiv") >= 2
        or author_list_score(text) >= 6
    )

chunks = [c for c in chunks if not looks_like_references(c.page_content)]
print("Chunks after removing references:", len(chunks))

Chunks after removing references: 3535


## 3. Embeddings & Vector Store
- **Embedding model:** `all-MiniLM-L6-v2` (384 dimensions). Small and fast, runs locally on 8 GB RAM, and the data is English-only.
- **Vector database:** ChromaDB with cosine similarity. Each chunk is stored with its text and metadata (`source`, `page`) so answers can cite them.
- **Persistence:** the store is saved to `backend/data/vector_store`, so the backend loads it directly without rebuilding.
- **Result:** 3535 chunks stored.

In [7]:
from sentence_transformers import SentenceTransformer
import chromadb, os

EMBED_MODEL = "all-MiniLM-L6-v2"
VS_DIR = "../backend/data/vector_store"
os.makedirs(VS_DIR, exist_ok=True)

model = SentenceTransformer(EMBED_MODEL)

client = chromadb.PersistentClient(path=VS_DIR)

try:
    client.delete_collection("papers")
except Exception:
    pass
collection = client.create_collection("papers", metadata={"hnsw:space": "cosine"})


texts = [c.page_content for c in chunks]
metas = [c.metadata for c in chunks]
ids = [f"chunk_{i}" for i in range(len(chunks))]

BATCH = 256
for i in range(0, len(texts), BATCH):
    emb = model.encode(texts[i:i+BATCH], normalize_embeddings=True).tolist()
    collection.add(
        ids=ids[i:i+BATCH],
        documents=texts[i:i+BATCH],
        metadatas=metas[i:i+BATCH],
        embeddings=emb,
    )

print("Stored chunks:", collection.count())

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Stored chunks: 3535


## 4. Retrieval & Prompting
`retrieve()` embeds the question with the same model and returns the top-k most similar chunks from ChromaDB, with source, page, and similarity score.

In [8]:
def retrieve(question, k=4):
    q_emb = model.encode([question], normalize_embeddings=True).tolist()
    res = collection.query(query_embeddings=q_emb, n_results=k * 3)
    hits, seen = [], set()
    for doc, meta, dist in zip(res["documents"][0], res["metadatas"][0], res["distances"][0]):
        key = (meta["source"], meta["page"])
        if key in seen:
            continue
        seen.add(key)
        hits.append({
            "text": doc,
            "source": meta["source"],
            "page": meta["page"],
            "score": round(1 - dist, 3),
        })
        if len(hits) == k:
            break
    return hits

In [9]:
for h in retrieve("What are large language models?"):
    print(h["source"], "| page", h["page"], "| score", h["score"])
    print(h["text"][:300], "\n")

2407.01505v1.pdf | page 11 | score 0.743
But I can confidently say that I have a deep understanding of the general principles and methodologies involved. Would you like to explore any specific aspect of large language model development in more detail? Perhaps you’re curious about the challenges of training models on biased data, or the eth 

2405.11357v3.pdf | page 1 | score 0.583
arXiv:2405.11357v3 [cs.CL] 23 Jul 2024 Large Language Models Lack Understanding of Character Composition of Words Andrew Shin 1 Kunitake Kaneko 1 Abstract Large language models (LLMs) have demonstrated remarkable performances on a wide range of natural language tasks. Y et, LLMs’ successes have been 

2306.06371v1.pdf | page 7 | score 0.574
Large Language Models (LLMs) - GPT-3 (Brown et al., 2020) marked the beginning of the era of Large Language Models (LLMs). It showed that, by scaling up language models, the few-shot learning is possible, and it can sometimes even reach competitive results in tasks such as

### Prompt & Generation
The prompt contains the numbered context chunks (with source and page) and the question. The model must answer only from the context, cite chunk numbers like [1], and say it could not find the answer if the context does not contain it. Temperature is 0 to reduce hallucination.

In [10]:
import ollama, re

LLM_MODEL = "llama3.2:3b"

def build_prompt(question, hits):
    context = "\n\n".join(
        f"[{i+1}] (source: {h['source']}, page {h['page']})\n{h['text']}"
        for i, h in enumerate(hits)
    )
    return f"""You are a document assistant. Answer the question using ONLY the context below.
Rules:
- If the answer is not in the context, say exactly: "I couldn't find this in the documents."
- Cite sources inline ONLY as [1], [2] (the chunk numbers). Do not write file names or page numbers.
- Do not use outside knowledge.
- Reuse the wording of the context as much as possible.
- Use only the context chunks that directly answer the question. Ignore unrelated chunks.
- Keep the answer short (at most 3 sentences).

Context:
{context}

Question: {question}
Answer:"""

def ngrams(text, n=4):
    w = re.findall(r"\w+", text.lower())
    return {tuple(w[i:i+n]) for i in range(len(w) - n + 1)}

def attribute(answer, hits, min_overlap=2):
    a = ngrams(answer)
    scored = sorted(
        ((len(a & ngrams(h["text"])), i) for i, h in enumerate(hits)),
        reverse=True,
    )
    return [i for c, i in scored if c >= min_overlap][:3]

def ask(question, k=6):
    hits = retrieve(question, k)
    if not hits or hits[0]["score"] < 0.45:
        return {"answer": "I couldn't find this in the documents.", "sources": []}
    prompt = build_prompt(question, hits)
    resp = ollama.chat(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": 0, "num_ctx": 4096},
    )
    answer = resp["message"]["content"]
    answer = re.sub(r"\s*\((?:source|sources)\s*:[^)]*\)", "", answer)
    answer = re.sub(r"(?:according to|as (?:mentioned|stated|shown) in)\s*(?:figure \d+ in\s*)?\[\d+\](?:\s*(?:,|and)\s*\[\d+\])*,?\s*", "", answer, flags=re.I)
    answer = re.sub(r"\s*(?:,\s*)?(?:as (?:mentioned|stated|shown) in|see)\s*\[\d+\]", "", answer)
    answer = re.sub(r"\s*\[\d+\]", "", answer)
    answer = re.sub(r",\.", ".", answer)
    answer = re.sub(r"\s+([.,])", r"\1", answer).strip()
    answer = answer[:1].upper() + answer[1:]

    if "couldn't find this in the documents" in answer:
        return {"answer": answer, "sources": []}

    idx = attribute(answer, hits)
    if not idx:  
        return {"answer": "I couldn't find this in the documents.", "sources": []}
    tags = "".join(f"[{j+1}]" for j in range(len(idx)))
    sources = [f"[{j+1}] {hits[i]['source']} (page {hits[i]['page']})" for j, i in enumerate(idx)]
    return {"answer": f"{answer} {tags}", "sources": sources}

    idx = attribute(answer, hits) or list(range(len(hits)))  
    tags = "".join(f"[{j+1}]" for j in range(len(idx)))
    sources = [f"[{j+1}] {hits[i]['source']} (page {hits[i]['page']})" for j, i in enumerate(idx)]
    return {"answer": f"{answer} {tags}", "sources": sources}
result = ask("What is reinforcement learning?")
print(result["answer"])
print(*result["sources"], sep="\n")

Reinforcement learning is a computational term for a successful class of algorithms solving Markov Decision Processes by sampling and learning from data. [1]
[1] 1705.05172v1.pdf (page 3)


In [11]:
for q in ["What is reinforcement learning?", "What are large language models?"]:
    r = ask(q)
    print(r["answer"])
    print(*r["sources"], sep="\n")
    print()

Reinforcement learning is a computational term for a successful class of algorithms solving Markov Decision Processes by sampling and learning from data. [1]
[1] 1705.05172v1.pdf (page 3)

Large language models are a type of model that has demonstrated remarkable performances on a wide range of natural language tasks. They have been shown to be successful in tasks concerning words, sentences, or documents, but it remains questionable how much they understand the minimal units of text, namely characters. [1]
[1] 2405.11357v3.pdf (page 1)



In [12]:
questions = [
    "What is reinforcement learning?",
    "What is the Q-learning update equation?",
    "What is meta-reinforcement learning?",
    "How does emotion influence learning in RL agents?",
    "What are large language models?",
    "What is a transformer architecture?",
    "What is retrieval-augmented generation?",
    "What is a convolutional neural network used for?",
    "Who won the 2022 FIFA World Cup?",
    "What is the capital of France?",
]

eval_rows = []
for q in questions:
    r = ask(q)
    eval_rows.append({"question": q, "answer": r["answer"], "sources": r["sources"]})
    print("Q:", q)
    print("A:", r["answer"])
    print(*r["sources"], sep="\n")
    print("-" * 80)

Q: What is reinforcement learning?
A: Reinforcement learning is a computational term for a successful class of algorithms solving Markov Decision Processes by sampling and learning from data. [1]
[1] 1705.05172v1.pdf (page 3)
--------------------------------------------------------------------------------
Q: What is the Q-learning update equation?
A: The Q-learning update equation is given by: Q(s,a ) =Q(s,a ) +α [ r(s,a,s′) +γ max a′ Q(s′,a′)−Q(s,a ) ] (4) where α specifies a learning rate. [1]
[1] 1705.05172v1.pdf (page 7)
--------------------------------------------------------------------------------
Q: What is meta-reinforcement learning?
A: Meta-reinforcement learning (meta-RL) considers a family of machine learning methods that learn to reinforcement learn. That is, meta-RL methods use sample-inefficient machine learning to learn sample-efficient reinforcement learning algorithms, or components thereof. As such, meta-RL is a special case of meta-learning, with the property that 

In [14]:
new_qs = [
    "What is federated learning?",
    "What is masked language modeling?",
    "What is a skip connection in a neural network?",
    "How is BERT pre-trained?",
    "How do I bake sourdough bread?",
    "Who is the president of Egypt?",
]
for q in new_qs:
    r = ask(q)
    print("Q:", q)
    print("A:", r["answer"])
    print(*r["sources"], sep="\n")
    print("-" * 80)

Q: What is federated learning?
A: Federated learning is the basic approach for collaborative learning on distributed datasets, where the training of agents is federated through a common round-based protocol coordinated by a central server, as shown in Figure 5. The key idea of federated learning is to federate the training of agents through a common protocol, allowing each agent to receive the jointly trained model, which can be used to perform inference independently on local data. This framework is used to improve the learning effectiveness of models used by agents to minimize their respective local objectives. [1][2]
[1] 2609.02984v1.pdf (page 8)
[2] 2609.02984v1.pdf (page 14)
--------------------------------------------------------------------------------
Q: What is masked language modeling?
A: Masked Language Modeling (MLM) is a method that uses an encoder only in the Transformer neural network architecture. It consists in selecting randomly a set of positions from the input token

## 5. Evaluation
Results on 10 test questions (7 answerable from the corpus, 3 not answerable: Q7, Q9, Q10). Verdicts were checked manually against the retrieved chunks.

| # | Question | Retrieved source | Answer | Correct? |
|---|---|---|---|---|
| 1 | What is reinforcement learning? | 1705.05172v1, p.3 | Class of algorithms solving MDPs by sampling and learning from data | Yes (checked against the source text) |
| 2 | What is the Q-learning update equation? | 1705.05172v1, p.7 | Equation (4) with the learning rate α, matches the source | Yes |
| 3 | What is meta-reinforcement learning? | 2301.08028v4, p.6 | Family of ML methods that learn to reinforcement learn, special case of meta-learning | Yes (checked against the source text) |
| 4 | How does emotion influence learning in RL agents? | 1705.05172v1, p.25-26 | Higher rewards and faster learning, with the source's own citations | Yes |
| 5 | What are large language models? | 2405.11357v3, p.1 | Models that perform well on many NLP tasks | Yes, but reflects the angle of one paper (character understanding) |
| 6 | What is a transformer architecture? | 1706.03762, p.3 | Stacked self-attention encoder-decoder, N=6 layers with two sub-layers | Mostly (checked): matches the encoder description; the answer applies "two sub-layers" to the whole architecture, and the retrieved chunk ends before the decoder's layers are described |
| 7 | What is retrieval-augmented generation? | none | Refused | Correct refusal (term only appears in a framework name and a reference) |
| 8 | What is a convolutional neural network used for? | 2301.00942v1, p.53-54 | Image semantic segmentation, image synthesis | Partial: mixes descriptions of different architectures on the same page |
| 9 | Who won the 2022 FIFA World Cup? | none | Refused | Correct refusal |
| 10 | What is the capital of France? | none | Refused | Correct refusal |


### New questions (not used for tuning)
Six extra questions asked after all thresholds and prompts were fixed (4 answerable, 2 out of scope).

| # | Question | Retrieved source | Answer | Correct? |
|---|---|---|---|---|
| 11 | What is federated learning? | 2609.02984v1, p.8, 14 | Collaborative learning on distributed data coordinated by a central server | Mostly (checked): the first two sentences match p.8; the last sentence credits the base framework with what p.14 says about its variations |
| 12 | What is masked language modeling? | 2306.06371v1, p.6 | Randomly masks input tokens and predicts them | Partial: confuses the encoder-only architecture with the MLM objective |
| 13 | What is a skip connection in a neural network? | 2301.00942v1, p.29 | Adds a layer's input to its output | Yes (checked against the source text); the equation is partly garbled by PDF extraction |
| 14 | How is BERT pre-trained? | 1810.04805, p.3 | Trained on unlabeled data over pre-training tasks | Partial: correct but incomplete. The section describing the two tasks (p.4) was ranked 8th, outside the top 6 |
| 15 | How do I bake sourdough bread? | none | Refused | Correct refusal |
| 16 | Who is the president of Egypt? | none | Refused | Correct refusal |

### Failure cases and mitigations
1. **Reference lists crowded out real content.** Author lists and bibliographies matched question words and filled the top results. Mitigation: chunks that look like references are removed before embedding. The first version of the rule deleted a real introduction chunk (meta-RL), so it was tightened.
2. **The 3B model cited wrong sources** (mixed up section and page numbers, wrong chunk numbers). Mitigation: sources are computed in code by matching 4-word phrases between the answer and each chunk. If nothing matches, the assistant refuses instead of showing unrelated sources.
3. **The transformer question could not be answered (Q6).** The corpus had no paper explaining it, and the answer built from unrelated chunks was ungrounded. Mitigation: a similarity threshold (top score < 0.45 means refuse), and adding the papers "Attention Is All You Need" and BERT to the corpus. The threshold comes from only 10 questions.
4. **Page headers leaked into answers (Q4).** The model turned a header ("26 Moerland et al.") into an invented citation. Mitigation: repeated header/footer lines are removed before cleaning.
5. **The right chunk was ranked 5th (Q1).** Retrieving 4 chunks missed the definition. Mitigation: retrieve 6 chunks, one per page (overlap produced duplicates).
6. **Remaining limits.** The small 3B model sometimes mixes sentences from neighbouring text (Q8, Q12) and answers from the angle of a single paper (Q5). On new questions, the right chunk can be ranked below the top 6 (Q14). Prompt wording is fragile: adding "reuse the wording of the context" fixed Q5 but made Q1 copy from an unrelated chunk, so it was balanced with "use only chunks that directly answer".

## 6. Export
The persisted Chroma store is in `backend/data/vector_store`. The settings the backend must reuse are saved in `backend/data/rag_config.json`.

In [13]:
import json

config = {
    "embedding_model": EMBED_MODEL,
    "collection_name": "papers",
    "distance": "cosine",
    "chunk_size": 1000,
    "chunk_overlap": 200,
    "top_k": 6,
    "similarity_threshold": 0.45,
    "llm_model": LLM_MODEL,
    "num_ctx": 4096,
    "num_chunks": collection.count(),
}
with open("../backend/data/rag_config.json", "w") as f:
    json.dump(config, f, indent=2)
print(config)

{'embedding_model': 'all-MiniLM-L6-v2', 'collection_name': 'papers', 'distance': 'cosine', 'chunk_size': 1000, 'chunk_overlap': 200, 'top_k': 6, 'similarity_threshold': 0.45, 'llm_model': 'llama3.2:3b', 'num_ctx': 4096, 'num_chunks': 3535}
